In [ ]:
import sys

import pandas as pd
import numpy as np

sys.path.append(r'C:\Users\user\Desktop\github\sotrans_analytics_department')
from source.core.de.serving.datamart import Datamart

# ETL

## Extract

In [ ]:
datamart: Datamart = Datamart(
    period=8,
    agg_cat_cols=(
        'wh_name',
        'ca_name',
        'ca_type',
        'ca_status',
        'brand_name_1c',
        'sku_id_1c',
    ),
    agg_dgt_cols=(
        'cnt',
    ),
    agg_dt_cols=(
        'year',
        'week'
    )
)

In [ ]:
# ???????? ???????;
datamart.IEnd.add_final()

In [ ]:
# ??????????? ????? ?????????;
datamart.ITransfer.add_transfer()

In [ ]:
# ?????? ?? ??????? ???????;
datamart.IArrival.add_arrival()

In [ ]:
# ?????? ?? ????????;
datamart.IExpend.add_sale()

## Transform

### Pre-Processing

In [ ]:
# ?????? ????????? ? ?????????????? ????????;
for col in (
        'wh_name',
        'ca_name',
        'ca_type',
        'ca_status',
        'brand_name_1c',
        'sku_id_1c'
):
    datamart.df[col] = datamart.df[col].fillna(value='_??? ??????')

In [ ]:
# ?????? ????????? ? ?????????????? ????????;
for col in (
        'en_final_cnt_fd',
        'tr_transfer_cnt_fd',
        'ar_arrival_cnt_fd',
        'ex_sale_cnt_sum_8_week',
):
    datamart.df[col] = datamart.df[col].fillna(value=0)

In [ ]:
# ????????????? ?????/???????;
condition = (
    datamart.df['ca_type']
    .isin(values=['????????? ????????', '???????? ??????? ? ??'])
)

datamart.df['ca_type'] = (
    np.where(
        condition,
        '?????????',
        '???????'
    )
)

In [ ]:
# ??????????????? ??????????;
datamart.df = datamart.df.query(
    expr="ca_type == '?????????'"
    " and wh_name != '_?? ????????????'"
)

In [ ]:
datamart.df = (
    datamart.df
    .groupby(
        by=[
            'wh_name',
            'brand_name_1c',
            'sku_id_1c'
        ]
    )
    .aggregate(
        {
            'en_final_cnt_fd': 'sum',
            'tr_transfer_cnt_fd': 'sum',
            'ar_arrival_cnt_fd': 'sum',
            'ex_sale_cnt_sum_8_week': 'sum'
        }
    )
    .reset_index()
)

In [ ]:
# ?????? ?????????? ?????? ? ????????;
datamart.IHaving.add_stock(
    agg_cat_cols=(
        'wh_name',
        'brand_name_1c',
        'sku_id_1c'
    )
)

In [ ]:
# ??????????????? ??????????;
datamart.df = datamart.df.query(
    expr="wh_name != '_?? ????????????'"
)

In [ ]:
(
    datamart.df
    .query(expr='sku_id_1c == "??293430"')
)

### Feature Engineering

In [ ]:
# ?????? ???????????? ??????? ? ?????? ??????????? ? ??????? ???????;
datamart.df['fact_cnt'] = (
    datamart.df['en_final_cnt_fd']
    + datamart.df['tr_transfer_cnt_fd']
    + datamart.df['ar_arrival_cnt_fd']
)

In [ ]:
(
    datamart.df
    .query(expr='sku_id_1c == "??293430"')
)

In [ ]:
datamart.df['mean_month_sales'] = [
    sales / stock * 4.5 
    if stock > 0
    else sales / 25 * 4.5
    for stock, sales in zip(
        datamart.df['hv_stock_8_week'],
        datamart.df['ex_sale_cnt_sum_8_week']
    )
]

In [ ]:
(
    datamart.df
    .query(expr='sku_id_1c == "??293430"')
)

In [ ]:
datamart.df['move_to_spb'] = [
    stock - mean_sales
    if stock > mean_sales
    else 0
    for stock, mean_sales in zip(
        datamart.df['fact_cnt'],
        datamart.df['mean_month_sales']
    )
]

In [ ]:
(
    datamart.df
    .query(expr='sku_id_1c == "??293430"')
)

### Separating

In [ ]:
# ??????????????? DataFrame; ?????? ??????????? ?????;
df_wh_spb: pd.DataFrame = datamart.df.query(
    expr="wh_name == '_????? ?????-????????? (??????? ???)'"
)

In [ ]:
(
    df_wh_spb
    .query(expr='sku_id_1c == "??293430"')
)

In [ ]:
# ??????????????? DataFrame; ?????? ??????????? ?????;
df_shops: pd.DataFrame = datamart.df.query(
    expr="wh_name != '_????? ?????-????????? (??????? ???)' "
    "and wh_name != '_?? ????????????'"
)

### Group by

In [ ]:
pt_wh_spb: pd.DataFrame = (
    df_wh_spb
    .groupby(by=['brand_name_1c', 'sku_id_1c'])
    .aggregate(
        {
            'en_final_cnt_fd': 'sum',
            'tr_transfer_cnt_fd': 'sum',
            'ar_arrival_cnt_fd': 'sum',
            'ex_sale_cnt_sum_8_week': 'sum',
            'hv_stock_8_week': 'mean',
            'fact_cnt': 'sum',
            'mean_month_sales': 'sum'
        }
    )
    .rename(
        columns={
            'ex_sale_cnt_sum_8_week': 'spb_ex_sale_cnt_sum_8_week',
            'mean_month_sales': 'spb_mean_month_sales',
            'hv_stock_8_week': 'spb_hv_stock_8_week',
            'fact_cnt': 'spb_fact_cnt',
        }
    )
    .reset_index()
)

In [ ]:
(
    pt_wh_spb
    .query(expr='sku_id_1c == "??293430"')
)

In [ ]:
pt_shops: pd.DataFrame = (
    df_shops
    .groupby(by=['wh_name', 'brand_name_1c', 'sku_id_1c'])
    .aggregate(
        {
            'ex_sale_cnt_sum_8_week': 'sum',
            'hv_stock_8_week': 'mean',
            'fact_cnt': 'sum',
            'mean_month_sales': 'sum',
            'move_to_spb': 'sum'
        }
    )
    .rename(
        columns={
            'ex_sale_cnt_sum_8_week': 'shop_ex_sale_cnt_sum_8_week',
            'mean_month_sales': 'shop_mean_month_sales',
            'hv_stock_8_week': 'shop_hv_stock_8_week',
            'fact_cnt': 'shop_fact_cnt',
        }
    )
    .reset_index()
)

In [ ]:
result_table: pd.DataFrame = pd.merge(
    left=pt_wh_spb,
    right=pt_shops,
    how='outer',
    on=['brand_name_1c', 'sku_id_1c']
)

In [ ]:
(
    result_table
    .query(expr='sku_id_1c == "??323186"')
)

In [ ]:
result_table

## Load

In [ ]:
result_table.to_json(path_or_buf='move_to_wh_spb.json', orient='table', index=False)